# flatshapes: train the region classifier

A tiny CNN (about 0.2 M parameters, under 1 MB) that tells from a region mask which shape it is, so
`detect_shapes` fits only that shape model. Trained on synthetic scenes from `tools/bench.py`; every step
writes to Google Drive, so a session reset loses nothing: run all cells again and it continues.

Runtime: **GPU (T4)** is enough; data generation runs on the CPU.

Result: `region_classifier.onnx` + `region_classifier.json` in the Drive folder. Copy both into
`flatshapes/flatshapes/models/` on your computer (or commit them to the repo) and `detect_shapes` uses them
automatically (`pip install onnxruntime` once).

In [ ]:
# --- settings ---
DRIVE_DIR = "/content/drive/MyDrive/flatshapes_classifier"  # data, checkpoints and the model go here
SCENES = 20000          # synthetic scenes (about 11 regions each)
VAL_SCENES = 1000       # extra scenes with other seeds, only for checking
EPOCHS = 10
DATA_VERSION = "v2"     # v2: text, bows, ribbons, blobs, balloons on strings as "not a shape"; pitted shapes
BATCH = 512
REPO = "https://github.com/ElenaKu/flatshapes"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## Get the code
If the repository is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon on the left; a fine-grained token with read access to the repo is enough).

In [ ]:
import subprocess
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/content/flatshapes"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/flatshapes"], check=True)
else:
    subprocess.run(["git", "-C", "/content/flatshapes", "pull"], check=True)
print("code ready")

## Make the training data (resumable, chunks saved to Drive)

In [ ]:
import subprocess, sys

def run(args):
    """Run a script and show its output live (plain subprocess.run output does not appear in Colab)."""
    p = subprocess.Popen([sys.executable, "-u"] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed with exit code {p.returncode}: {' '.join(args)}")

data_dir = f"{DRIVE_DIR}/data_{DATA_VERSION}"
val_dir = f"{DRIVE_DIR}/val_{DATA_VERSION}"
run(["/content/flatshapes/tools/region_dataset.py", data_dir, "--scenes", str(SCENES), "--seed", "1000", "--chunk", "500"])
run(["/content/flatshapes/tools/region_dataset.py", val_dir, "--scenes", str(VAL_SCENES), "--seed", "900000", "--chunk", "500"])
print("data ready")


In [ ]:
import glob, numpy as np, collections
sys.path.insert(0, "/content/flatshapes")
from flatshapes.classifier import CLASSES, SIZE

def load_dir(d):
    X, y = [], []
    for f in sorted(glob.glob(f"{d}/regions_*.npz")):
        z = np.load(f)
        X.append(z["X"]); y.append(z["y"])
    return np.concatenate(X), np.concatenate(y).astype(np.int64)

Xtr, ytr = load_dir(data_dir)
Xva, yva = load_dir(val_dir)
print("train", Xtr.shape, "val", Xva.shape)
print({CLASSES[k]: v for k, v in sorted(collections.Counter(ytr.tolist()).items())})

## Model and training (checkpoint saved to Drive after every epoch)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, time
dev = "cuda" if torch.cuda.is_available() else "cpu"

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class RegionNet(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        self.f = nn.Sequential(block(2, 16), nn.MaxPool2d(2),    # 32
                               block(16, 32), nn.MaxPool2d(2),   # 16
                               block(32, 64), nn.MaxPool2d(2),   # 8
                               block(64, 96), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(96, n_cls)
    def forward(self, x):
        return self.head(torch.flatten(self.f(x), 1))

model = RegionNet(len(CLASSES)).to(dev)
print(sum(p.numel() for p in model.parameters()), "parameters")

# class weights: rare classes count a little more
cnt = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float32)
w = torch.tensor((cnt.sum() / np.maximum(cnt, 1)) ** 0.5, dtype=torch.float32, device=dev)
w = w / w.mean()

Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr)
Xva_t = torch.from_numpy(Xva).float().div(255).to(dev); yva_t = torch.from_numpy(yva).to(dev)

def augment(x):
    # shape kinds do not change under flips and quarter turns
    if torch.rand(1) < 0.5: x = x.flip(3)
    if torch.rand(1) < 0.5: x = x.flip(2)
    k = int(torch.randint(0, 4, (1,)))
    return torch.rot90(x, k, (2, 3)) if k else x

opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
steps = EPOCHS * ((len(Xtr) + BATCH - 1) // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=steps)
ckpt = f"{DRIVE_DIR}/checkpoint_{DATA_VERSION}.pt"
start = 0
old = f"{DRIVE_DIR}/checkpoint.pt"
if not os.path.exists(ckpt) and os.path.exists(old):
    # warm start from the first model: it already knows the shapes, it only has to learn the new "not a shape" cases
    model.load_state_dict(torch.load(old, map_location=dev)["model"])
    print("started from the previous model's weights")
if os.path.exists(ckpt):
    c = torch.load(ckpt, map_location=dev)
    model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
    start = c["epoch"] + 1
    print("resumed after epoch", c["epoch"] + 1)

def evaluate():
    model.eval(); correct = 0; conf = np.zeros((len(CLASSES), len(CLASSES)), int)
    with torch.no_grad():
        for i in range(0, len(Xva_t), 2048):
            p = model(Xva_t[i:i + 2048]).argmax(1)
            for t, q in zip(yva_t[i:i + 2048].tolist(), p.tolist()):
                conf[t, q] += 1
    return conf.trace() / conf.sum(), conf

for ep in range(start, EPOCHS):
    model.train(); t0 = time.time(); perm = torch.randperm(len(Xtr_t)); tot = 0.0
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i + BATCH]
        x = augment(Xtr_t[idx].to(dev).float().div(255)); y = ytr_t[idx].to(dev)
        loss = F.cross_entropy(model(x), y, weight=w, label_smoothing=0.05)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(idx)
    acc, _ = evaluate()
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep), ckpt)
    print(f"epoch {ep + 1}/{EPOCHS}  loss {tot / len(perm):.4f}  val accuracy {acc:.4f}  {time.time() - t0:.0f} s")

## Check: which classes get mixed up

In [ ]:
acc, conf = evaluate()
print(f"val accuracy {acc:.4f}\n")
print("true \\ predicted".ljust(12) + "".join(c[:6].rjust(7) for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(c[:11].ljust(12) + "".join(str(v).rjust(7) for v in conf[i]), f"  recall {conf[i, i] / max(1, conf[i].sum()):.3f}")

## Export to ONNX (copy both files into flatshapes/flatshapes/models/)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)  # ONNX export needs the onnx package
import json
model.eval().cpu()
onnx_path = f"{DRIVE_DIR}/region_classifier.onnx"
torch.onnx.export(model, torch.zeros(1, 2, SIZE, SIZE), onnx_path, input_names=["regions"], output_names=["logits"],
                  dynamic_axes={"regions": {0: "n"}, "logits": {0: "n"}}, opset_version=17, dynamo=False)
with open(f"{DRIVE_DIR}/region_classifier.json", "w") as fh:
    json.dump({"classes": CLASSES, "size": SIZE, "val_accuracy": float(acc), "scenes": SCENES}, fh, indent=1)
print("saved", onnx_path, os.path.getsize(onnx_path) // 1024, "KB")

## Optional: does it make detection faster?
Runs the benchmark with and without the classifier (CPU, like on your computer).

In [ ]:
import shutil
os.makedirs("/content/flatshapes/flatshapes/models", exist_ok=True)
shutil.copy(onnx_path, "/content/flatshapes/flatshapes/models/region_classifier.onnx")
shutil.copy(f"{DRIVE_DIR}/region_classifier.json", "/content/flatshapes/flatshapes/models/region_classifier.json")
run(["-m", "pip", "install", "-q", "onnxruntime"])
run(["/content/flatshapes/tools/bench.py", "--workers", "0", "--no-classifier", "--save", f"{DRIVE_DIR}/bench_rules.json"])
run(["/content/flatshapes/tools/bench.py", "--workers", "0", "--save", f"{DRIVE_DIR}/bench_classifier.json",
     "--compare", f"{DRIVE_DIR}/bench_rules.json"])